# שלב 07 — זיהוי קהילות (Community Detection)

אלגוריתם **Louvain** מוצא קהילות — קבוצות תחנות שמחוברות צפוף יותר בתוכן מאשר ביניהן. השאלה: האם הקהילות שנוצרות מהמבנה בלבד תואמות אזורים גיאוגרפיים אמיתיים? **Modularity** מודד עד כמה החלוקה לקהילות 'טובה' (ערך גבוה = קהילות ברורות).

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy networkx matplotlib seaborn python-bidi python-louvain

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain

def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
GRAPH_DIR = BASE / "outputs" / "02_graph_construction"
METRICS_CSV = BASE / "outputs" / "04_centrality_analysis" / "stop_metrics.csv"
OUT_DIR = BASE / "outputs" / "07_community_detection"
FIG_DIR = BASE / "figures" / "07_community_detection"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## הרצת Louvain ו-Label Propagation

שני אלגוריתמים על הרכיב הקשור הגדול, עם seed=42 לשחזור.

In [ ]:
with open(GRAPH_DIR / "graph_undirected.pkl", "rb") as f:
    G = pickle.load(f)
metrics = pd.read_csv(METRICS_CSV, encoding="utf-8-sig")
print(f"גרף: {G.number_of_nodes():,} צמתים, {G.number_of_edges():,} קשתות")

Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
partition = community_louvain.best_partition(Gc, weight="weight", random_state=42)
modularity = community_louvain.modularity(partition, Gc, weight="weight")
print(f"Louvain: {len(set(partition.values()))} קהילות, Modularity = {modularity:.4f}")

communities = list(nx.algorithms.community.label_propagation_communities(Gc))
partition_lp = {node: i for i, comm in enumerate(communities) for node in comm}
print(f"Label Propagation: {len(communities)} קהילות")

## שמירת שיוכי הקהילות וסיכום

לכל תחנה שומרים את הקהילה שלה, ולכל קהילה מחשבים גודל, קשתות פנימיות ומרכז גיאוגרפי.

In [ ]:
assign_rows = [{
    "stop_id": n, "stop_name": G.nodes[n].get("stop_name", ""),
    "lat": G.nodes[n].get("lat"), "lon": G.nodes[n].get("lon"),
    "region": G.nodes[n].get("region", ""),
    "community_louvain": partition.get(n, -1), "community_lp": partition_lp.get(n, -1),
} for n in G.nodes()]
assign_df = pd.DataFrame(assign_rows)
assign_df.to_csv(OUT_DIR / "community_assignments.csv", index=False, encoding="utf-8-sig")

rows = []
for cid in sorted(set(partition.values())):
    members = [n for n, c in partition.items() if c == cid]
    sub = G.subgraph(members)
    lats = [G.nodes[n].get("lat") for n in members if G.nodes[n].get("lat")]
    lons = [G.nodes[n].get("lon") for n in members if G.nodes[n].get("lon")]
    rows.append({"community_id": cid, "size": len(members), "internal_edges": sub.number_of_edges(),
                 "centroid_lat": round(np.mean(lats), 4) if lats else None,
                 "centroid_lon": round(np.mean(lons), 4) if lons else None})
summary = pd.DataFrame(rows).sort_values("size", ascending=False)
summary.to_csv(OUT_DIR / "community_summary.csv", index=False, encoding="utf-8-sig")
print(f"{len(summary)} קהילות")
summary.head()

## גרף: מפת הקהילות

כל תחנה צבועה לפי הקהילה שלה. אם הצבעים יוצרים אזורים גיאוגרפיים רציפים — הקהילות תואמות גיאוגרפיה.

In [ ]:
comm_ids = list(set(partition.values()))
cmap = matplotlib.colormaps.get_cmap("tab20").resampled(len(comm_ids))
color_map = {cid: cmap(i) for i, cid in enumerate(comm_ids)}
lons, lats, cols = [], [], []
for node, cid in partition.items():
    lat, lon = G.nodes[node].get("lat"), G.nodes[node].get("lon")
    if lat and lon:
        lons.append(lon); lats.append(lat); cols.append(color_map[cid])

fig, ax = plt.subplots(figsize=(8, 11))
ax.scatter(lons, lats, s=3, color=cols, alpha=0.5, linewidths=0)
ax.set_title(f"מפת קהילות — Louvain (Modularity={modularity:.3f})")
ax.set_xlabel("קו אורך"); ax.set_ylabel("קו רוחב")
plt.tight_layout()
plt.savefig(FIG_DIR / "community_map_louvain.png", dpi=150)
plt.show()

top = summary.head(20).sort_values("size")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top["community_id"].astype(str), top["size"], color="#7c3aed")
ax.set_xlabel("מספר תחנות"); ax.set_title("גדלי הקהילות הגדולות (Top 20)")
plt.tight_layout()
plt.savefig(FIG_DIR / "community_sizes_bar.png", dpi=150)
plt.show()